<a href="https://colab.research.google.com/github/arelkeselbri/gsi073/blob/main/aula4_Finetune_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4 — Fine-tuning de um Transformer Decoder-only

Nesta prática faremos um **fine-tuning supervisionado (SFT)** em um **decoder-only causal LM** usando o menor modelo *útil* e popular para demos no Hugging Face (https://huggingface.co/distilbert/distilgpt2):

- **Modelo base:** `distilbert/distilgpt2`

Passos:
1. Carregar/definir documentos (toy ou pasta local)
2. Tokenização + *grouping* em blocos (CLM)
3. Fine-tuning com `Trainer`
4. Inferência

In [32]:
# (Opcional) Instalar dependências no ambiente local/Colab
!pip -q install transformers datasets accelerate


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch: 2.10.0+cpu
cuda available: False


## 1) Dataset — Documentos (toy) ou pasta local

### Opção A (didática): documentos toy
### Opção B (real): ler `.txt` de uma pasta local

> Para aula, a Opção A roda rápido e mostra o pipeline completo.


In [34]:
# Opção A: documentos toy
#
#docs = [
 #   "UFU forma estudantes de IA aplicada para muitas áreas.",
#
 #   "Um transformer decoder-only prevê o próximo token com auto-regressão.",
  #  "Fine-tuning ajusta um modelo pré-treinado para um domínio específico.",
   # "Temperatura controla aleatoriedade; top-k e top-p controlam o corte do vocabulário.",
    #"A Universidade Federal de Uberlândia é top demais.",
    #"A Universidade Federal de Uberlândia é top demais mesmo."
 #]

#ds = Dataset.from_dict({"text": docs}).train_test_split(test_size=0.1, seed=42)
#ds

In [28]:
 #Opção B (descomente se quiser ler arquivos .txt de uma pasta):
from glob import glob
paths = glob("*.txt")
docs = []
for p in paths:
    text = open(p, "r", encoding="utf-8").read()
    print(text)
    examples = text.split("\n\n")  
    docs.extend(examples)
print("Número de exemplos:", len(docs))
ds = Dataset.from_dict({"text": docs}).train_test_split(test_size=0.1, seed=42)

Um RPG de fantasia épica onde o jogador é o último Guardião de um reino assolado por uma Sombra ancestral. Suas escolhas definem o destino dos deuses e mortais.

Corridas de arcade em alta velocidade em paisagens urbanas futuristas iluminadas por néon, com customização profunda de carros e trilha sonora synthwave cativante.

Gerencie uma colônia em um planeta alienígena hostil. Construa, pesquise e defenda seus colonos contra a fauna perigosa e mistérios cósmicos.

Um jogo de terror psicológico em primeira pessoa. Explore uma mansão abandonada enquanto desvenda um mistério de família macabro e evita aparições fantasmagóricas.

Um relaxante simulador de forja em pixel art. Colete materiais, aprimore suas ferramentas e crie armas lendárias sob encomenda para heróis e vilões.

Um jogo de plataforma 2D com a mecânica de manipulação temporal. O protagonista pode saltar para trás ou para frente no tempo para resolver quebra-cabeças complexos.

Um RPG de ação focado em combate tático de espad

## 2) Tokenizer + Modelo base

Usaremos o tokenizer do modelo base e definiremos `pad_token`, pois GPT-2 não tem por padrão.


In [29]:
model_id = "distilbert/distilgpt2"

tok = AutoTokenizer.from_pretrained(model_id)

# GPT-2 não define pad_token por padrão; necessário para batches com padding
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print("Vocab size:", len(tok))
print("Model loaded:", model_id)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Vocab size: 50257
Model loaded: distilbert/distilgpt2


## 3) Tokenização + agrupamento em blocos (Causal LM)

Em *Causal Language Modeling*, a label é o próprio `input_ids` deslocado internamente pelo modelo.
Uma forma padrão é concatenar tudo e quebrar em blocos fixos (`block_size`).


In [30]:
block_size = 64  # pequeno para rodar rápido

def tokenize_fn(batch):
    return tok(batch["text"], truncation=True)

tok_ds = ds.map(tokenize_fn, batched=False, remove_columns=["text"])

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    print('total_len = ', total_length)
    total_length = (total_length // block_size) * block_size
    print('total_len = ', total_length)
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_ds = tok_ds.map(group_texts, batched=True)
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

total_len =  3932
total_len =  3904


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

total_len =  397
total_len =  384


## 4) Fine-tuning (SFT) com `Trainer`

> Com o seguinte código, abstraímos a parte do treino e executar 10 épocas para o modelo aprender o conteúdo do novo corpus.


In [32]:
args = TrainingArguments(
    output_dir="./ft_tiny_gpt2",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=10,
    learning_rate=5e-4,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_ds["train"],
    eval_dataset=lm_ds["test"],
    data_collator=collator
)

trainer.train()

C:\Users\Dell\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.974535,4.473640
2,3.807399,4.434610
3,2.434765,4.618690
4,1.496432,4.804733
5,1.188279,5.475190
6,1.296551,6.467906
7,0.475476,6.929555
8,0.197867,7.319566
9,0.092331,7.659441
10,0.229285,7.730574


TrainOutput(global_step=310, training_loss=1.5966236383924561, metrics={'train_runtime': 105.9828, 'train_samples_per_second': 5.756, 'train_steps_per_second': 2.925, 'total_flos': 9961938616320.0, 'train_loss': 1.5966236383924561, 'epoch': 10.0})

## 5) Inferência depos do finetune



In [33]:
prompt=tok("Um vasto reino", return_tensors = "pt", truncation = True).input_ids[0,]
inputs = prompt.unsqueeze(0).to(device)
with torch.no_grad():
    out_ids = model.generate(
        inputs,
        max_new_tokens=10,
        do_sample=True,
        pad_token_id=tok.eos_token_id,
        temperature= 0.7,
        top_k = 50,
        top_p = 1.0
    )
print(f"Inference: {tok.decode(out_ids[0])}")

Inference: Um vasto reino dos deuses, gerenciando o cal


In [ ]:
print("\n--- Inferência em Documentos de Treino ---")
train_docs = ds['train']['text']

for i, doc_text in enumerate(train_docs):
    initial_tokens = tok(doc_text, return_tensors="pt", truncation=True).input_ids[0, :10]
    inputs = initial_tokens.unsqueeze(0).to(device) # unsqueeze to add batch dimension

    with torch.no_grad():
        out_ids = model.generate(
            inputs,
            max_new_tokens=10,
            do_sample=True,
            pad_token_id=tok.eos_token_id,
            temperature= 0.7,
            top_k = 50,
            top_p = 1.0
        )

    # Decode the sequence
    # prompt_text = tok.decode(initial_tokens, skip_special_tokens=True)
    # Essa é a parte de uma sinopse de um jogo chamado "Hollow Knight"
    # é do domínio dos jogos mas não está presente no txt 
    prompt_text = "um vasto reino subterrâneo de insetos"
    generated_text = tok.decode(out_ids[0], skip_special_tokens=True)
    print(f"doc{i}: {prompt_text}*{generated_text[len(prompt_text):]}*")



--- Inferência em Documentos de Treino ---
doc0: um vasto reino subterrâneo* planeta alienígena hostil.*
doc1: um vasto reino subterrâneo*atraindo aventureiros cor*
doc2: um vasto reino subterrâneo* descobrir a verdade sobre*
doc3: um vasto reino subterrâneo*âneas, onde você treina*
doc4: um vasto reino subterrâneo*o de dinossauros. Comande um*
doc5: um vasto reino subterrâneo*Subaute contra chefes mitológ*
doc6: um vasto reino subterrâneo*retrônia em um mundo mágico*
doc7: um vasto reino subterrâneo*ante com um deserto perigoso. Compre uma*
doc8: um vasto reino subterrâneo*forja em pixel artesanato onde você compra,*
doc9: um vasto reino subterrâneo*um mundo mágico em constante mudan*
doc10: um vasto reino subterrâneo*ista. Pilote um arrastão, gerenci*
doc11: um vasto reino subterrâneo*temático de dinossauros.*
doc12: um vasto reino subterrâneo*eça. Organize livros encantados, a*
doc13: um vasto reino subterrâneo*tmosférico. Lidere*
doc14: um vasto reino subterrâneo* em turnos em um 

# Exercício prático

Escolha um prompt que não estava nos dados de treino, mas que seja do mesmo domínio. Faça comparações. Documente a saída do modelo base (sem treino) e do modelo ajustado. Verifique se o modelo "aprendeu" termos técnicos ou o vocabulário específico que você forneceu nos arquivos .txt.

Sem realizar o treino, somente com a base, gerou umas respostas nada a ver, não sei se fiz algo errado
---------------------
doc0: um vasto reino subterrâneo de insetos*eu colô*
doc1: um vasto reino subterrâneo de insetos*dépense de*
doc2: um vasto reino subterrâneo de insetos**
doc3: um vasto reino subterrâneo de insetos*
*
doc4: um vasto reino subterrâneo de insetos*este, di*
doc5: um vasto reino subterrâneo de insetos*, short story, a*
doc6: um vasto reino subterrâneo de insetos*e los que llamos y*
doc7: um vasto reino subterrâneo de insetos* dar razu en el mundo.*
doc8: um vasto reino subterrâneo de insetos*xela la métologie.
------------------------------